# Phase 4: Data Preprocessing & Augmentation

This notebook covers:
1. Splitting the raw data into Train (70%), Validation (10%), and Test (20%) sets.
2. Defining the PyTorch dataset transforms (including data augmentation for training).
3. Creating `DataLoader` objects.
4. Visualizing a batch of augmented training data.

In [ ]:
import os
import random
import shutil
from pathlib import Path

random.seed(42)

# Paths
RAW_DATA_DIR = Path('../data/raw')
PROCESSED_DATA_DIR = Path('../data/processed')

CLASSES = ['open', 'closed']
SPLITS = {'train': 0.7, 'val': 0.1, 'test': 0.2}

# Create processed directories
for split in SPLITS.keys():
    for cls in CLASSES:
        (PROCESSED_DATA_DIR / split / cls).mkdir(parents=True, exist_ok=True)

print("Created directory structure for processed data.")

In [ ]:
def split_data(cls):
    src_dir = RAW_DATA_DIR / cls
    images = [f for f in os.listdir(src_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.webp'))]
    random.shuffle(images)
    
    n_total = len(images)
    n_train = int(n_total * SPLITS['train'])
    n_val = int(n_total * SPLITS['val'])
    
    train_imgs = images[:n_train]
    val_imgs = images[n_train:n_train + n_val]
    test_imgs = images[n_train + n_val:]
    
    for split_name, img_subset in zip(['train', 'val', 'test'], [train_imgs, val_imgs, test_imgs]):
        dest_dir = PROCESSED_DATA_DIR / split_name / cls
        for img in img_subset:
            shutil.copy2(src_dir / img, dest_dir / img)
            
    print(f"{cls.capitalize()}: {len(train_imgs)} train | {len(val_imgs)} val | {len(test_imgs)} test (Total: {n_total})")

# Clear existing files in processed to avoid mixed files if run multiple times
for split in SPLITS.keys():
    for cls in CLASSES:
        folder = PROCESSED_DATA_DIR / split / cls
        for f in os.listdir(folder):
            os.remove(folder / f)

# Distribute images
for cls in CLASSES:
    split_data(cls)

### Creating DataLoaders with Transforms

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# ResNet34 Standard Normalization
mean = [0.485, 0.456, 0.406] 
std = [0.229, 0.224, 0.225]

# Data Augmentation for Training
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)), # Resize + Crop
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

# Standard pre-processing for Validation / Test (No Augmentation)
eval_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

# Map datasets
image_datasets = {
    'train': datasets.ImageFolder(root=PROCESSED_DATA_DIR / 'train', transform=train_transforms),
    'val': datasets.ImageFolder(root=PROCESSED_DATA_DIR / 'val', transform=eval_transforms),
    'test': datasets.ImageFolder(root=PROCESSED_DATA_DIR / 'test', transform=eval_transforms)
}

# Create DataLoaders
BATCH_SIZE = 32

dataloaders = {
    x: DataLoader(image_datasets[x], batch_size=BATCH_SIZE, shuffle=(x == 'train'), num_workers=0)
    for x in ['train', 'val', 'test']
}

dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val', 'test']}
class_names = image_datasets['train'].classes

print(f"Classes: {class_names}")
print(f"Dataset Sizes: {dataset_sizes}")

### Visualizing Augmented Training Data

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torchvision

def imshow(inp, title=None):
    """Imshow for Tensor."""
    inp = inp.numpy().transpose((1, 2, 0))
    # Reverse normalization
    inp = std * inp + mean
    inp = np.clip(inp, 0, 1)
    plt.figure(figsize=(12, 12))
    plt.imshow(inp)
    if title is not None:
        plt.title(title)
    plt.pause(0.001)  # pause a bit so that plots are updated

# Get a batch of training data
inputs, classes = next(iter(dataloaders['train']))

# Display 8 images
out = torchvision.utils.make_grid(inputs[:8])
imshow(out, title=[class_names[x] for x in classes[:8]])